# 存续股量额第三源核对

## tl;dr

33个原大幅差异均获BaoStock对TDX的数值支持；同花顺缓存与库一致。不是全市场验收，不改原数据，不重算策略。

## Context & Methods

固定原20只存续对照，2025-11-26至12-02；股票×日期×字段。仅重算登记容差内的数值对齐。

### Key Assumptions

第三供应商未证明独立上游；当前缓存与当前安装代码不能证明历史载荷/版本。后续收益没有参与检验。

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
root = Path.cwd().parents[1]
out = root / 'output/rps_input_audit_20261007/v1'
registration = json.loads((out/'registration.json').read_text(encoding='utf-8'))
p = registration['protocol']
print(p['window'], len(p['codes']), p['liquidity_tolerance'])

['2025-11-26', '2025-12-02'] 20 {'rtol': 1e-06, 'atol': 1}


## Data

保存的原始BaoStock响应与请求参数在baostock目录；同花顺只读旧导出、当前下载缓存、TDX未复权窗口均保留CSV。复跑此笔记不联网、不修改数据库。

In [2]:
keys = ['ts_code','trade_date']
tables = {name: pd.read_csv(out/f'{name}_window.csv', dtype={'trade_date':str}).set_index(keys) for name in ['hithink','tdx','dump','baostock']}
for name, frame in tables.items():
    assert frame.index.is_unique
    assert len(frame) == 100
    assert set(frame.index.get_level_values('ts_code')) == set(p['codes'])
print({name: len(frame) for name, frame in tables.items()})

{'hithink': 100, 'tdx': 100, 'dump': 100, 'baostock': 100}


## Results

下面从四份窗口表直接重算，不使用预先生成的布尔标记。容差用于数值一致性，不能作为选股阈值。

In [3]:
focus = pd.read_csv(out/'prior_large_cells_recheck.csv', dtype={'trade_date':str})
assert len(focus) == 33
counts = dict(tdx_bao=0, hithink_bao=0, hithink_dump=0)
for row in focus.itertuples():
    key = (row.ts_code, row.trade_date)
    values = {name: frame.loc[key, row.field] for name, frame in tables.items()}
    assert all(pd.notna(v) for v in values.values())
    for label, left, right in [('tdx_bao','tdx','baostock'), ('hithink_bao','hithink','baostock'), ('hithink_dump','hithink','dump')]:
        counts[label] += int(np.isclose(values[left], values[right], **p['liquidity_tolerance']))
assert counts == {'tdx_bao':33,'hithink_bao':0,'hithink_dump':33}
print(counts)
display(focus.loc[focus.ts_code.eq('000001.SZ'), ['ts_code','trade_date','field','hithink','tdx','dump','baostock']])

{'tdx_bao': 33, 'hithink_bao': 0, 'hithink_dump': 33}


,ts_code,trade_date,field,hithink,tdx,dump,baostock
0,000001.SZ,2025-11-27,vol,10118471.0,1.011847e+08,10118471.0,1.011847e+08
1,000001.SZ,2025-12-01,vol,10373227.0,1.037323e+08,10373227.0,1.037323e+08
2,000001.SZ,2025-11-27,amount,11783089.0,1.178309e+09,11783089.0,1.178309e+09
3,000001.SZ,2025-12-01,amount,12054803.0,1.205480e+09,12054803.0,1.205480e+09


## Takeaways

数据关尚未关闭：下一步需要有来源凭证的隔离修正及信号影响核查。不能猜乘倍率，也不能把修正后的历史叫独立样本外。这里只做一个小范围精确对账，表格比趋势图更适合核对字段。